In [50]:
import os
from dotenv import load_dotenv
#load variables from .env   
load_dotenv()

#access variables
g_api_key = os.getenv("G_API_KEY")

from typing import Dict, List, Optional
import nest_asyncio
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext, Tool
from pydantic_ai.settings import ModelSettings
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

provider = GoogleProvider(api_key=g_api_key)
model = GoogleModel('gemini-2.5-flash', provider=provider)


In [ ]:
import requests
import json
import pandas as pd
import nest_asyncio

In [3]:
class Record(BaseModel):
    """Structured response."""
    wave_max: float
    wave_min: float
    wind_speed: float
    wind_direction: str
    date_time: str

In [ ]:
class SpotDetails(BaseModel):
    """Structure for incoming spots query."""

    spot_id_surfline: str
    name: str
    best_conditions: str
    best_tide: str


In [71]:
Spot_list=[]
spot1=SpotDetails(
    spot_id_surfline="6418e27e6478f02820fac1e6",
    name="Sopelana",best_conditions="Less than 2 m in wave height. If wind speed is greater then 10, offshore conditions is best, however if wind speed is less then 10 any wind direction is fine",
    best_tide="Best tide conditions from 2 hours before to 2 hours after each high tide")
Spot_list.append(spot1)

In [6]:
class TableOutput(BaseModel):
    rows: List[Record]

In [7]:
#Define function to convert unix timestamp to date and time

import datetime

def unix_to_datetime(unix_ts: int) -> str:
    """
    Convert a Unix timestamp (seconds) to ISO 8601 datetime string (UTC).
    """
    dt = datetime.datetime.fromtimestamp(unix_ts, tz=datetime.timezone.utc)
    return dt.isoformat()

In [ ]:
nest_asyncio.apply()

In [ ]:
#Agent preparation
agent1 = Agent(
    model=model,
    output_type=TableOutput,
    tools=[unix_to_datetime,surfline_wave,surfline_wind],
    system_prompt=(
        "You are an intelligent agent to support my choice for the surfing spot. "
        "Your role is to report strictly extracted data, do not hallucinate fabricating new data"
        "Access only this website https://services.surfline.com/kbyg/spots/forecasts/surf?cacheEnabled=true&days=10&intervalHours=1&spotId=6418e27e6478f02820fac1e6&units[waveHeight]=M and this website https://services.surfline.com/kbyg/spots/forecasts/wind?spotId=6418e27e6478f02820fac1e6&days=10&intervalHours=1&corrected=false&cacheEnabled=true&units%5BwindSpeed%5D=KTS to extrapolate wave and wind informations about this specific spot which is Sopelana" 
        "Report hourly, reporting max wave, min wave, wind speed and direction for each relevant datapoint"
        "You may call tools when needed. "
        "Use unix_to_datetime to convert Unix timestamps."
        "Use data_from_surfline_wave to extract wave data from URL"
        "Use data_from_surfline_wind to extract wind data from URL"
        "It is good to surf with offshore winds and wves under 2 meters, please filter using those parameters." 
    ), 
)
    # These are known when writing the code

In [ ]:
#Agent preparation
agent2 = Agent(
    model=model,
    #output_type=TableOutput,
    tools=[unix_to_datetime],
    deps_type=SpotDetails,
    system_prompt=(
        """
        You are an intelligent agent to support my choice for the surfing spot.
        Your role is to report when the date and times when the tide or the wave conditions are optimal for surfing the desired spot in your dependencies.
        You can only report data you have retrieved with tools.
        You can extract the spotID from your dependencies.
        Use the surfline_tide tool to fetch tide data spot data, you will need to imput the spotID to the function. 
        Use the surfline_wave tool to fetch waves data spot data, you will need to imput the spotID to the function. 
        Use the surfline_wind tool to fetch wind data spot data, you will need to imput the spotID to the function.
        You shall not report Unix timestamps, use the unix_to_datetime tool to convert them."""
    ), 
    retries=2,
    model_settings=ModelSettings(
        max_tokens=8000,
        temperature=0.1
    )
)


@agent2.tool_plain(retries=1)
async def surfline_tide(spot_id: str)->list:
    """
    Inputs a Spot ID, than it generates the desired URL, then it fetches for the tides information for date forecast from a surfline.com URL, it outputs a list of dictionaries. 
    """
    print(spot_id)

    URL=f"https://services.surfline.com/kbyg/spots/forecasts/tides?spotId={spot_id}&days=11&cacheEnabled=true&units%5BtideHeight%5D=M"

    print(f"Calling URL: {URL}")

    raw_data_tide=requests.get(URL)
    raw_dict_tide=json.loads(raw_data_tide.text)
    raw_dict_tide_data=raw_dict_tide['data']
    raw_list_tide=raw_dict_tide_data['tides']
    return raw_list_tide

@agent2.tool_plain(retries=1)
async def surfline_wave(spot_id: str)->list:
    """
    Generate the desired URL, then it fetches for the waves information for date forecast from a surfline.com URL, it outputs a list of dictionaries. 
    """
    URL=f"https://services.surfline.com/kbyg/spots/forecasts/surf?cacheEnabled=true&days=10&intervalHours=1&spotId={spot_id}&units[waveHeight]=M"

    print(f"Calling URL: {URL}")

    raw_data_surf=requests.get(URL)
    raw_dict_surf=json.loads(raw_data_surf.text)
    raw_dict_surf_data=raw_dict_surf['data']
    raw_list_surf=raw_dict_surf_data['surf']
    return raw_list_surf

@agent2.tool_plain(retries=1)
async def surfline_wind(spot_id: str)->list:
    """
    Generate the desired URL, then it fetches for the wind information for date forecast from a surfline.com URL, it outputs a list of dictionaries. 
    """
    URL=f"https://services.surfline.com/kbyg/spots/forecasts/wind?spotId={spot_id}&days=10&intervalHours=1&corrected=false&cacheEnabled=true&units%5BwindSpeed%5D=KTS"
    
    print(f"Calling URL: {URL}")
    
    raw_data_wind=requests.get(URL)
    raw_dict_wind=json.loads(raw_data_wind.text)
    raw_dict_wind_data=raw_dict_wind['data']
    raw_list_wind=raw_dict_wind_data['wind']
    return raw_list_wind

In [68]:
# Add dynamic system prompt based on dependencies
@agent2.system_prompt
async def add_spot_details(ctx: RunContext[SpotDetails]) -> str:
    return f"Customer details: {ctx.deps}"  # These depend in some way on context that isn't known until runtime


In [69]:
response2=agent2.run_sync("Can you tell me when there will be the best condition for surfing in sopelana in the next three days? ",deps=spot1)


6418e27e6478f02820fac1e6
Calling URL: https://services.surfline.com/kbyg/spots/forecasts/tides?spotId=6418e27e6478f02820fac1e6&days=11&cacheEnabled=true&units%5BtideHeight%5D=M
Calling URL: https://services.surfline.com/kbyg/spots/forecasts/surf?cacheEnabled=true&days=10&intervalHours=1&spotId=6418e27e6478f02820fac1e6&units[waveHeight]=M
Calling URL: https://services.surfline.com/kbyg/spots/forecasts/wind?spotId=6418e27e6478f02820fac1e6&days=10&intervalHours=1&corrected=false&cacheEnabled=true&units%5BwindSpeed%5D=KTS


In [70]:
response2.output

"The optimal conditions for surfing in Sopelana in the next three days are:\n\n*   **2026-03-04T02:00:00+00:00 UTC**: Wave height up to 2m, wind speed 7.26 knots (any direction is fine as it's below 10 knots), and within the optimal high tide window.\n*   **2026-03-04T03:00:00+00:00 UTC**: Wave height up to 1.8m, wind speed 7.89 knots (any direction is fine as it's below 10 knots), and within the optimal high tide window.\n*   **2026-03-04T04:00:00+00:00 UTC**: Wave height up to 1.8m, wind speed 8.36 knots (any direction is fine as it's below 10 knots), and within the optimal high tide window.\n*   **2026-03-04T10:00:00+00:00 UTC**: Wave height up to 1.5m, wind speed 7.60 knots (any direction is fine as it's below 10 knots), and within the optimal high tide window.\n*   **2026-03-04T11:00:00+00:00 UTC**: Wave height up to 1.4m, wind speed 6.28 knots (any direction is fine as it's below 10 knots), and within the optimal high tide window.\n*   **2026-03-04T19:00:00+00:00 UTC**: Wave heig

In [ ]:
##response.output.model_dump()
#response.output.rows[0].model_dump()
#df = pd.DataFrame([row.model_dump() for row in response.output.rows])
#df